In [ ]:
import math
import pandas as pd
from collections import Counter


def entropy(y):

    counts = Counter(y)
    total = len(y)

    ent = 0

    for count in counts.values():

        probability = count / total

        ent -= probability * math.log2(probability)

    return ent


def information_gain(data, feature, target):

    parent_entropy = entropy(data[target])

    values = data[feature].unique()

    weighted_child_entropy = 0

    for value in values:

        child = data[data[feature] == value]

        weight = len(child) / len(data)

        child_entropy = entropy(child[target])

        weighted_child_entropy += weight * child_entropy

    gain = parent_entropy - weighted_child_entropy

    return gain


def find_best_feature(data, features, target):

    best_feature = None
    best_gain = -1

    for feature in features:

        gain = information_gain(
            data,
            feature,
            target
        )

        if gain > best_gain:

            best_gain = gain
            best_feature = feature

    return best_feature


def build_tree(data, features, target):

    if len(data[target].unique()) == 1:

        return data[target].iloc[0]

    if len(features) == 0:

        return data[target].mode()[0]

    best_feature = find_best_feature(
        data,
        features,
        target
    )

    tree = {
        best_feature: {}
    }

    remaining_features = [
        feature
        for feature in features
        if feature != best_feature
    ]

    for value in data[best_feature].unique():

        child = data[
            data[best_feature] == value
        ]

        subtree = build_tree(
            child,
            remaining_features,
            target
        )

        tree[best_feature][value] = subtree

    return tree


def predict_one(tree, row):

    if not isinstance(tree, dict):

        return tree

    feature = next(iter(tree))

    value = row[feature]

    if value in tree[feature]:

        return predict_one(
            tree[feature][value],
            row
        )

    return None


def predict(data, tree):

    predictions = []

    for _, row in data.iterrows():

        predictions.append(
            predict_one(tree, row)
        )

    return predictions


def accuracy(y_true, y_pred):

    correct = 0

    for actual, predicted in zip(
        y_true,
        y_pred
    ):

        if actual == predicted:

            correct += 1

    return correct / len(y_true)


data = pd.DataFrame({

    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rain",
        "Rain", "Rain", "Overcast", "Sunny",
        "Sunny", "Rain", "Sunny", "Overcast",
        "Overcast", "Rain"
    ],

    "Temperature": [
        "Hot", "Hot", "Hot", "Mild",
        "Cool", "Cool", "Cool", "Mild",
        "Cool", "Mild", "Mild", "Mild",
        "Hot", "Mild"
    ],

    "Humidity": [
        "High", "High", "High", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "High"
    ],

    "Wind": [
        "Weak", "Strong", "Weak", "Weak",
        "Weak", "Strong", "Strong", "Weak",
        "Weak", "Weak", "Strong", "Strong",
        "Weak", "Strong"
    ],

    "PlayTennis": [
        "No", "No", "Yes", "Yes",
        "Yes", "No", "Yes", "No",
        "Yes", "Yes", "Yes", "Yes",
        "Yes", "No"
    ]
})


features = [
    "Outlook",
    "Temperature",
    "Humidity",
    "Wind"
]

target = "PlayTennis"


print("Information Gain:")

for feature in features:

    gain = information_gain(
        data,
        feature,
        target
    )

    print(
        feature,
        "=",
        round(gain, 4)
    )


tree = build_tree(
    data,
    features,
    target
)

print("\nDecision Tree:")
print(tree)


predictions = predict(
    data,
    tree
)

print("\nPredictions:")
print(predictions)


acc = accuracy(
    data[target],
    predictions
)

print("\nAccuracy:",
      round(acc * 100, 2),
      "%")